In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

import nltk
import spacy
import re
from load_data import load_novels

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nlp = spacy.load('en_core_web_sm')

df = load_novels(ROOT / "data" / "raw")

sample_sentence = "Mr. D'Arcy couldn't believe it—it was a 19th-century masterpiece, wasn't it? The price was $50.00!"

In [3]:
regex_tokens = re.findall(r'\b\w+\b', sample_sentence)
print("Regex Tokens:", regex_tokens)

nltk_tokens = nltk.word_tokenize(sample_sentence)
print("NLTK Tokens:", nltk_tokens)

spacy_tokens = [token.text for token in nlp(sample_sentence)]
print("spaCy Tokens:", spacy_tokens)


Regex Tokens: ['Mr', 'D', 'Arcy', 'couldn', 't', 'believe', 'it', 'it', 'was', 'a', '19th', 'century', 'masterpiece', 'wasn', 't', 'it', 'The', 'price', 'was', '50', '00']
NLTK Tokens: ['Mr.', "D'Arcy", 'could', "n't", 'believe', 'it', '—', 'it', 'was', 'a', '19th-century', 'masterpiece', ',', 'was', "n't", 'it', '?', 'The', 'price', 'was', '$', '50.00', '!']
spaCy Tokens: ['Mr.', "D'Arcy", 'could', "n't", 'believe', 'it', '—', 'it', 'was', 'a', '19th', '-', 'century', 'masterpiece', ',', 'was', "n't", 'it', '?', 'The', 'price', 'was', '$', '50.00', '!']


The regex tokenizer (\b\w+\b) is the crudest: it discards all punctuation and shatters
words at every non-word character — "couldn't" → "couldn"+"t", "D'Arcy" → "D"+"Arcy", "50.00" → "50"+"00". 
NLTK's Treebank tokenizer is linguistically informed: it keeps "Mr.", "D'Arcy" and "50.00" 
intact and correctly separates the clitic "n't", while treating "19th-century" as one token. 
spaCy agrees on contractions and punctuation but additionally splits "19th-century" into "19th"+"-"+"century", 
giving the most fine-grained, linguistically consistent segmentation of the three.

In [5]:
raw = df['raw_text'].iloc[0]

# Pattern 1: boilerplate/watermark stripper (Gutenberg header)
p1 = r'\*\*\*.*?START OF.*?\*\*\*'
n1 = len(re.findall(p1, raw, flags=re.IGNORECASE | re.DOTALL))
story = re.sub(p1, '', raw, flags=re.IGNORECASE | re.DOTALL)
print(f"Pattern 1 (boilerplate header): {n1} match(es) removed")

# Pattern 2: dialogue extractor (curly + straight quotes)
p2 = r'[“"]([^”"]*)[”"]'
dialogue = re.findall(p2, story)
print(f"Pattern 2 (dialogue): {len(dialogue)} quoted spans in the novel")
if dialogue:
    print("  example:", dialogue[0][:80])

# Pattern 3: metadata stripper (standalone page/chapter numbers)
p3 = r'^\s*\d+\s*$'
nums = re.findall(p3, story, flags=re.MULTILINE)
print(f"Pattern 3 (standalone numbers): {len(nums)} match(es) in novel")
demo = "text line\n  42  \nmore text\n17\n"
print("  pattern demo on synthetic lines:", re.findall(p3, demo, flags=re.MULTILINE))

# Extra checks: original spec + teammate's PDF pipeline
clean_ws = re.sub(r'\s+', ' ', story)
print("Whitespace normalization applied (original spec pattern).")
print("OceanofPDF watermark matches:", len(re.findall(r'OceanofPDF\.com', raw, flags=re.IGNORECASE)))

Pattern 1 (boilerplate header): 1 match(es) removed
Pattern 2 (dialogue): 2778 quoted spans in the novel
  example: the Captain,
Pattern 3 (standalone numbers): 0 match(es) in novel
  pattern demo on synthetic lines: ['  42  ', '17\n']
Whitespace normalization applied (original spec pattern).
OceanofPDF watermark matches: 0


Pattern 1: \*\*\*.*?START OF.*?\*\*\* (IGNORECASE | DOTALL) non-greedily matches and removes the Project Gutenberg header — the boilerplate "watermark" of our corpus — exactly 1 match per file, so license text never pollutes the vocabulary.
Pattern 2: [“"]([^”"]*)[”"] captures spans between curly or straight double quotes, extracting 2,778 dialogue spans from A Tale of Two Cities — the raw material for the project's future character-relationship ("gossip") analysis.
Pattern 3: ^\s*\d+\s*$ (MULTILINE) targets lines containing only a number, i.e. stray page numbers from PDF extraction; it finds 0 in our clean Gutenberg text but correctly catches "42" and "17" in the synthetic demo, so it is kept as a robustness guard in the cleaning pipeline.

Tokenizing 19th-century novels raises several challenges visible in our outputs. (1) Contractions: "couldn't" must become "could"+"n't"; regex gets this wrong while NLTK and spaCy handle it correctly. (2) Hyphenated compounds: "19th-century" is one token for NLTK but three for spaCy — neither choice is universally right, and the decision changes vocabulary size and downstream matching. (3) Meaningful punctuation: em-dashes ("it—it") and quotation marks must be separated from words without destroying dialogue boundaries, which is why our dialogue pattern accepts both curly and straight quotes. (4) Boilerplate and artifacts: every file opens with a Gutenberg license header and irregular whitespace that would distort frequency counts if not stripped; PDF-extracted editions of the same novels would additionally carry watermarks, stray page numbers, and OCR errors such as "l" for "I" or "â€™" for an apostrophe. Together these issues motivate the cleaning pipeline (header stripping, whitespace normalization, number-line removal) before any tokenization or modeling.